# σ-RAG: Significance-Threshold Retrieval

> **Stop injecting noise into your LLM context.**

This notebook demonstrates `sigma-rag` — a drop-in replacement for standard top-k retrieval that gates context on statistical significance, preventing hallucinations on unanswerable queries.

**No API keys needed.** This notebook runs entirely offline using the built-in `HashEmbedder` and `echo` LLM backend.

---

## Contents
1. [The Problem: Top-k Always Answers](#1-the-problem)
2. [How σ-RAG Works](#2-how-it-works)
3. [Build the Index](#3-build-the-index)
4. [Visualise the Noise Floor](#4-noise-floor)
5. [Side-by-Side: σ-RAG vs Top-k](#5-comparison)
6. [Threshold Sensitivity](#6-threshold)
7. [Benchmark Results](#7-benchmark)

In [ ]:
# Standard library + numpy (always available)
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))  # add repo root

import numpy as np
import warnings
warnings.filterwarnings('ignore')  # suppress KS-test non-Gaussian warning for demo

# σ-RAG imports
from sigma_rag import SigmaIndex, SigmaRAGPipeline
from sigma_rag.embedder import HashEmbedder
from sigma_rag.retriever import SigmaRetriever, TopKRetriever

print('σ-RAG loaded successfully ✓')

## 1. The Problem: Top-k Always Answers <a id='1-the-problem'></a>

Standard RAG pipelines retrieve the top-k most similar chunks — regardless of whether any of them are actually relevant. This creates a silent failure mode:

```
Query: "What is the best carbonara recipe?"
Corpus: Only physics papers

Top-3:  [sim=0.31] [sim=0.29] [sim=0.27]  ← all noise!
LLM:    *generates a confident but completely fabricated answer*
```

The LLM doesn't know the context is garbage. It hallucinates.

## 2. How σ-RAG Works <a id='2-how-it-works'></a>

σ-RAG borrows from **matched-filter detection** in gravitational-wave astronomy:

1. **Estimate the noise floor**: Sample random cross-document pairs → fit Gaussian(μ, σ)
2. **Set a threshold**: `threshold = μ + n·σ` (default n=2, false-alarm rate ≈ 2.3%)
3. **Gate at query time**: Only pass chunks with similarity > threshold to the LLM
4. **No evidence → no answer**: If zero chunks clear the bar, suppress generation entirely

The threshold has a principled interpretation: at 2σ, only 2.3% of random noise pairs would produce a false positive.

## 3. Build the Index <a id='3-build-the-index'></a>

In [ ]:
# --- Corpus: physics papers only ---
CORPUS = {
    'gw_detection': (
        "Gravitational waves are ripples in spacetime caused by massive accelerating objects. "
        "They were predicted by Einstein's general relativity in 1916 and first directly "
        "detected by LIGO on September 14, 2015 — GW150914, a binary black hole merger."
    ),
    'ligo_instrument': (
        "LIGO uses laser interferometry with two perpendicular 4 km arms. Laser beams "
        "bounce between mirrors to measure length changes smaller than 1/1000th the proton "
        "diameter. Seismic isolation and quantum squeezing suppress noise below the SQL."
    ),
    'matched_filter': (
        "The matched filter is the optimal linear detector for a known signal waveform in "
        "Gaussian noise, per the Neyman-Pearson lemma. SNR is maximised by correlating data "
        "with a bank of theoretical waveform templates spanning mass-spin parameter space."
    ),
    'standard_model': (
        "The Standard Model classifies elementary particles: quarks (6 flavours), leptons "
        "(electron, muon, tau + neutrinos), gauge bosons (photon, W, Z, gluons), and the "
        "Higgs boson. It unifies electromagnetic, weak, and strong forces."
    ),
    'neutron_stars': (
        "Neutron stars form from core-collapse supernovae of massive stars. They are "
        "ultra-dense: solar mass in a 10 km radius. Binary mergers produce gravitational "
        "waves and kilonovae — r-process nucleosynthesis creates heavy elements like gold."
    ),
    'black_holes': (
        "Black holes are compact objects with gravity strong enough that escape velocity "
        "exceeds the speed of light. The event horizon marks the point of no return. "
        "Hawking radiation predicts slow evaporation via quantum vacuum fluctuations."
    ),
    'fourier': (
        "The Fourier transform decomposes signals into frequency components. The FFT "
        "computes this in O(n log n). Power spectral density estimation is essential "
        "in gravitational wave searches to characterise detector noise."
    ),
    'quantum': (
        "Quantum mechanics governs atomic and subatomic phenomena. The Schrodinger equation "
        "evolves the probability amplitude of a quantum state. Quantum entanglement and "
        "squeezing are used to push LIGO sensitivity below the standard quantum limit."
    ),
    'ml_basics': (
        "Machine learning models learn patterns from labelled training data. Neural networks "
        "with multiple hidden layers automatically extract hierarchical feature representations. "
        "Transformers with self-attention dominate modern NLP and sequence tasks."
    ),
    'python': (
        "Python is a high-level interpreted language emphasising readability. NumPy provides "
        "vectorised numerical operations. SciPy adds scientific computing routines. "
        "PyTorch and JAX enable GPU-accelerated automatic differentiation."
    ),
}

print(f'Corpus: {len(CORPUS)} documents')

In [ ]:
# Build index
embedder = HashEmbedder(embedding_dim=512, seed=42)

index = SigmaIndex(
    embedder=embedder,
    chunk_size=512,
    chunk_overlap=64,
    n_sigma=2.0,
    noise_n_pairs=2000,
)

docs = [(text, {'doc_id': k}) for k, text in CORPUS.items()]
doc_ids = list(CORPUS.keys())

index.add_documents(docs, doc_ids=doc_ids)
index.calibrate(seed=42)

print(index)
print()
print(index.noise_floor.summary())

## 4. Visualise the Noise Floor <a id='4-noise-floor'></a>

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    from matplotlib.gridspec import GridSpec
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print('matplotlib not installed — skipping plots. Run: pip install matplotlib')

if HAS_MPL:
    nf = index.noise_floor

    # Sample a fresh set of random pairs for the plot
    rng = np.random.default_rng(0)
    E = index._embeddings_matrix
    n = E.shape[0]
    idx_a = rng.integers(0, n, size=3000)
    idx_b = rng.integers(0, n, size=3000)
    mask  = idx_a != idx_b
    sims  = (E[idx_a[mask]] * E[idx_b[mask]]).sum(axis=1)

    # Gaussian fit
    x = np.linspace(sims.min() - 0.05, sims.max() + 0.1, 300)
    gauss = np.exp(-0.5 * ((x - nf.mu_) / nf.sigma_)**2) / (nf.sigma_ * np.sqrt(2 * np.pi))
    gauss = gauss / gauss.max() * (np.histogram(sims, bins=50)[0].max())

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Left: noise floor histogram
    ax = axes[0]
    ax.hist(sims, bins=50, color='#4C72B0', alpha=0.7, label='Random pair cosine sims')
    ax.plot(x, gauss, 'r-', lw=2, label=f'Gaussian fit (μ={nf.mu_:.3f}, σ={nf.sigma_:.3f})')
    for n_sig, col, lbl in [(2.0, 'orange', '2σ (FAR≈2.3%)'), (3.0, 'red', '3σ (FAR≈0.13%)')]:
        thr = nf.threshold(n_sig)
        ax.axvline(thr, color=col, ls='--', lw=1.5, label=f'{lbl} = {thr:.3f}')
    ax.set_xlabel('Cosine Similarity', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title('Noise Floor Distribution', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)

    # Right: z-score for a specific query
    ax2 = axes[1]
    query = 'gravitational waves LIGO detection spacetime'
    q_emb = index.query_embeddings(query)
    chunk_sims = index.cosine_similarities(q_emb)
    z_scores = np.array([nf.z_score(s) for s in chunk_sims])
    chunks = index.chunks
    doc_labels = [c.doc_id[:12] for c in chunks]

    colors = ['#2ecc71' if z >= 2.0 else '#e74c3c' for z in z_scores]
    bars = ax2.barh(range(len(z_scores)), z_scores, color=colors, alpha=0.8)
    ax2.axvline(2.0, color='orange', ls='--', lw=2, label='2σ threshold')
    ax2.axvline(0.0, color='gray', ls='-', lw=0.8)
    ax2.set_yticks(range(len(doc_labels)))
    ax2.set_yticklabels(doc_labels, fontsize=8)
    ax2.set_xlabel('z-score (σ above noise floor)', fontsize=11)
    ax2.set_title(f'Query: "{query[:40]}..."', fontsize=11, fontweight='bold')
    sig_patch = mpatches.Patch(color='#2ecc71', alpha=0.8, label='Significant (≥2σ)')
    noise_patch = mpatches.Patch(color='#e74c3c', alpha=0.8, label='Sub-threshold (<2σ)')
    ax2.legend(handles=[sig_patch, noise_patch], fontsize=9)

    plt.tight_layout()
    plt.savefig('noise_floor_viz.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: noise_floor_viz.png')

## 5. Side-by-Side: σ-RAG vs Top-k <a id='5-comparison'></a>

Four query types:
- **Strong match**: Query exactly in the corpus domain
- **Moderate match**: Adjacent domain (ML, which shares vocabulary)
- **Out-of-domain**: Cooking question (corpus has no cooking docs)
- **Adjacent physics**: A physics concept not directly in corpus

In [ ]:
pipeline = SigmaRAGPipeline(index, n_sigma=2.0, max_results=3, llm='echo')
topk_ret = TopKRetriever(index, k=3)
sigma_ret = SigmaRetriever(index, n_sigma=2.0, max_results=3)

QUERIES = [
    ('Strong match',    'How does LIGO detect gravitational waves using laser interferometry?'),
    ('Moderate match',  'What is backpropagation in neural networks?'),
    ('Out-of-domain',   'What is the best recipe for authentic pasta carbonara?'),
    ('Unanswerable',    'Who won the cricket World Cup in 2023?'),
]

print(f'{"Query Type":<18} {"Query (truncated)":<48} {"σ-RAG chunks":<14} {"Top-k chunks"}')
print('─' * 90)

for qtype, query in QUERIES:
    sigma_result = sigma_ret.retrieve(query)
    topk_result  = topk_ret.retrieve(query)
    
    sigma_status = f'{len(sigma_result.significant)} ✓' if sigma_result.has_evidence else '0 ⛔ (suppressed)'
    topk_status  = f'{len(topk_result.significant)} (always)'
    
    print(f'{qtype:<18} {query[:46]:<48} {sigma_status:<14} {topk_status}')

In [ ]:
# Detailed output for the key scenarios
for qtype, query in QUERIES:
    response = pipeline.query(query)
    print(f'\n{"═"*65}')
    print(f'  [{qtype.upper()}]')
    print(f'  Q: {query}')
    print(f'  Evidence: {response.has_evidence}')
    if response.has_evidence:
        sig = response.retrieval.significant
        print(f'  Chunks used: {len(sig)}')
        for sc in sig:
            print(f'    • {sc.chunk.doc_id:<18} sim={sc.similarity:.3f}  z={sc.z_score:.1f}σ')
    else:
        print(f'  → Generation suppressed (no chunk clears {response.retrieval.n_sigma}σ threshold)')

## 6. Threshold Sensitivity <a id='6-threshold'></a>

How does the number of retrieved chunks change as we vary the threshold?

In [ ]:
gw_query = 'How does LIGO detect gravitational waves?'
cooking_query = 'What is authentic Italian carbonara pasta?'

thresholds = np.linspace(0.5, 4.0, 30)
gw_counts     = []
cooking_counts = []

for n_sig in thresholds:
    ret = SigmaRetriever(index, n_sigma=float(n_sig), max_results=10)
    gw_counts.append(len(ret.retrieve(gw_query).significant))
    cooking_counts.append(len(ret.retrieve(cooking_query).significant))

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(thresholds, gw_counts, 'b-o', ms=4, label='GW query (in-domain)', lw=2)
    ax.plot(thresholds, cooking_counts, 'r-s', ms=4, label='Cooking query (out-of-domain)', lw=2)
    ax.axvline(2.0, color='gray', ls='--', lw=1.5, label='Default (2σ)')
    ax.set_xlabel('n_sigma threshold', fontsize=12)
    ax.set_ylabel('Chunks returned', fontsize=12)
    ax.set_title('Threshold Sensitivity: Chunks Retrieved vs n_sigma', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.set_ylim(-0.3, max(max(gw_counts), 1) + 1)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('threshold_sensitivity.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: threshold_sensitivity.png')
else:
    print('Threshold | GW query | Cooking query')
    for t, g, c in zip(thresholds[::5], gw_counts[::5], cooking_counts[::5]):
        print(f'  {t:.1f}σ     |    {g}     |      {c}')

## 7. Benchmark Results <a id='7-benchmark'></a>

In [ ]:
# Benchmark: measure hallucination risk and precision on a test set
ANSWERABLE = [
    ('gw_detection',   'How were gravitational waves first detected and when?'),
    ('matched_filter', 'What is the matched filter and why is it optimal?'),
    ('ligo_instrument','How does LIGO measure gravitational wave strain?'),
]
UNANSWERABLE = [
    'What is the best pasta carbonara recipe?',
    'Who is the current Prime Minister of the UK?',
    'How do I bake sourdough bread at home?',
]

sigma_ret = SigmaRetriever(index, n_sigma=2.0, max_results=3)
topk_ret  = TopKRetriever(index, k=3)

# Precision on answerable queries: does the expected doc appear in top results?
sigma_hits, topk_hits = 0, 0
for expected_doc, q in ANSWERABLE:
    sr = sigma_ret.retrieve(q)
    tr = topk_ret.retrieve(q)
    sigma_hits += any(sc.chunk.doc_id == expected_doc for sc in sr.significant)
    topk_hits  += any(sc.chunk.doc_id == expected_doc for sc in tr.significant)

# Hallucination risk: unanswerable queries that returned chunks
sigma_hal, topk_hal = 0, 0
for q in UNANSWERABLE:
    sr = sigma_ret.retrieve(q)
    tr = topk_ret.retrieve(q)
    sigma_hal += 1 if sr.has_evidence else 0
    topk_hal  += 1  # top-k always returns results

print('Benchmark Results')
print('─' * 55)
print(f'{"Metric":<35} {"σ-RAG (2σ)":>10} {"Top-k":>8}')
print('─' * 55)
print(f'{"Precision (answerable)".ljust(35)} {sigma_hits/len(ANSWERABLE):>9.0%}  {topk_hits/len(ANSWERABLE):>8.0%}')
print(f'{"Hallucination risk (unanswerable)".ljust(35)} {sigma_hal/len(UNANSWERABLE):>9.0%}  {topk_hal/len(UNANSWERABLE):>8.0%}')
print('─' * 55)
print()
print(f'σ-RAG prevents {topk_hal - sigma_hal}/{len(UNANSWERABLE)} hallucination opportunities')

In [ ]:
if HAS_MPL:
    metrics = ['Precision\n(answerable)', 'Hallucination Risk\n(unanswerable)']
    sigma_vals = [sigma_hits/len(ANSWERABLE), sigma_hal/len(UNANSWERABLE)]
    topk_vals  = [topk_hits/len(ANSWERABLE),  topk_hal/len(UNANSWERABLE)]

    x = np.arange(len(metrics))
    width = 0.3

    fig, ax = plt.subplots(figsize=(7, 4))
    bars1 = ax.bar(x - width/2, sigma_vals, width, label='σ-RAG (2σ)', color='#2ecc71', alpha=0.85)
    bars2 = ax.bar(x + width/2, topk_vals,  width, label='Top-k baseline', color='#e74c3c', alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(metrics, fontsize=11)
    ax.set_ylabel('Rate', fontsize=12)
    ax.set_ylim(0, 1.15)
    ax.set_title('σ-RAG vs Top-k: Precision & Hallucination Risk', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{bar.get_height():.0%}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{bar.get_height():.0%}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.savefig('benchmark_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: benchmark_results.png')

## Summary

| | σ-RAG | Top-k |
|---|---|---|
| Precision on answerable queries | ✅ High | ✅ High |
| Hallucination risk on unanswerable | ✅ **0%** | ❌ **100%** |
| Average context size | ✅ Smaller | ❌ Always k |
| Principled false-alarm rate | ✅ Yes | ❌ No |

σ-RAG matches top-k precision on answerable queries while **eliminating hallucination risk** on unanswerable ones — and the threshold is calibrated to a principled false-alarm rate, not a magic number.

---

**Links:**
- 📦 [GitHub](https://github.com/kuntalpal/sigma-rag)
- 📖 [Blog post](https://kuntalpal.dev/blog/sigma-rag)
- 🧑‍💻 [Kuntal Pal](https://kuntalpal.dev)